# Tutorial: Generating Stereo Audio from Text with the `audio` Module

This notebook provides a comprehensive guide to using the `audio.py` module, which is designed to generate stereo audio from conversational transcripts using Google Cloud Text-to-Speech. It supports distinct voices for different roles (e.g., customer and agent) and can save the output locally or directly to Google Cloud Storage (GCS).

## 1. Setup and Installation

Before we dive into the code, we need to set up our environment and install the necessary libraries. This module relies on `pydub` for audio manipulation, which in turn requires `ffmpeg` to be installed on your system. We'll also mock the `conidk.wrapper.speech` and `conidk.wrapper.storage` modules, as they are internal wrappers for Google Cloud services.

### Google Cloud Project Setup
1.  **Google Cloud Project ID:** You'll need an active Google Cloud Project ID. Replace `YOUR_PROJECT_ID` with your actual project ID.
2.  **Enable APIs:** Ensure the **Cloud Text-to-Speech API** is enabled in your Google Cloud Project.
3.  **Authentication:** If running locally, make sure you're authenticated (`gcloud auth application-default login`) or have `GOOGLE_APPLICATION_CREDENTIALS` set up for a service account. In Colab, your environment is usually authenticated through your Google account.

In [ ]:
# Install necessary Python libraries
!pip install pydub

# For Colab environments, install ffmpeg
!apt-get update && apt-get install -y ffmpeg

print("Installation complete.")

## 2. Mocking Internal Dependencies and Providing `audio.py` Source

The `audio.py` module depends on `conidk.wrapper.speech` and `conidk.wrapper.storage`. Since these are internal wrappers, we'll create mock implementations in this notebook to make the demonstration runnable without needing the actual `conidk` package. The original `audio.py` source code will then be provided in the following cell.

These mocks will print messages to simulate the behavior of calling Google Cloud Text-to-Speech and Google Cloud Storage, returning dummy audio or confirming uploads, respectively.

In [ ]:
import enum
import io
import logging
from typing import Optional

from pydub import AudioSegment #type: ignore
from IPython.display import Audio, display

# Suppress pydub's INFO logging for cleaner output
logging.getLogger('pydub').setLevel(logging.WARNING)

# --- MOCK IMPLEMENTATIONS FOR `conidk.wrapper` ---
# These mock classes simulate the behavior of the actual Google Cloud Text-to-Speech
# and Google Cloud Storage wrappers.

class MockTextToSpeech:
    """Mock class for conidk.wrapper.speech.TextToSpeech."""
    def __init__(self, project_id: str):
        self.project_id = project_id
        print(f"Mock TextToSpeech client initialized for project: {self.project_id}")

    def synthesize(self, text: str, voice: str, language_code: str, sample_rate_hertz: int) -> bytes:
        print(f"  [Mock TTS] Synthesizing: '{text[:50]}{'...' if len(text) > 50 else ''}'\n             (voice='{voice}', lang='{language_code}', sr='{sample_rate_hertz}')")
        # Create a silent WAV audio segment as a placeholder
        # Adjust duration based on text length to make it somewhat realistic
        duration_ms = max(500, len(text) * 75) # 75ms per character, min 500ms
        audio = AudioSegment.silent(duration=duration_ms, frame_rate=sample_rate_hertz)
        buffer = io.BytesIO()
        audio.export(buffer, format="wav")
        return buffer.getvalue()

class MockGcs:
    """Mock class for conidk.wrapper.storage.Gcs."""
    class ContentType(enum.Enum):
        WAV = "audio/wav"
        # Add other content types if needed by audio.py's storage interactions

    def __init__(self, bucket_name: str, project_id: str):
        self.bucket_name = bucket_name
        self.project_id = project_id
        print(f"Mock GCS client initialized for bucket='{self.bucket_name}', project='{self.project_id}'")

    def upload_blob(self, file_name: str, data: bytes, content_type: ContentType):
        print(f"  [Mock GCS] Successfully uploaded blob '{file_name}' to bucket '{self.bucket_name}'\n             (content type: {content_type.value}, size: {len(data)} bytes)")

# Create mock modules that replicate the expected import paths
class MockSpeechModule:
    TextToSpeech = MockTextToSpeech

class MockStorageModule:
    Gcs = MockGcs
    ContentType = MockGcs.ContentType

# Assign mocks to the names that audio.py expects to import
speech = MockSpeechModule
storage = MockStorageModule

# --- START OF ORIGINAL audio.py CODE (with modified imports for mocks) ---
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""A class to generate audio from text using Google Cloud Text-to-Speech."""

# Original imports would be:
# from conidk.wrapper import speech
# from conidk.wrapper import storage
# But we are using our mocks `speech` and `storage` defined above.


class Roles(enum.StrEnum):
    """Enum for supported roles"""

    AGENT = "AGENT"
    CUSTOMER = "CUSTOMER"


class VoiceRole(enum.Enum):
    """Enum for supported voices for each role"""

    CUSTOMER = "en-US-Wavenet-A"
    AGENT = "en-US-Wavenet-D"


class GenerateAudio:
    """
    A class to generate audio from text using Google Cloud Text-to-Speech.
    """

    def __init__(
        self,
        project_id: str,
    ):
        """
        Initializes the GenerateAudio class.
        Args:
            project_id (str): The Google Cloud project ID.
        """
        self.project_id = project_id
        logging.info(
            "Be advised: for this module to properly work 'ffmpeg' needs to be locally instaled"
        )

    def bulk(
        self,
        transcripts: list,
        audio_file_path: str,
    ) -> None:
        """
        Processes multiple JSON transcript file, generates stereo audio,
        and saves it.
        Args:
            transcripts (list): a list of JSON transcripts
            audio_file_path (str): local path to store the files
        """
        for transcript in transcripts:
            self.single(transcript=transcript, audio_file_path=audio_file_path)

    ##TO-DO break down in functions
    def single(
        self,
        transcript,
        audio_file_path: str,
        language: Optional[str] = "en-US",
        agent_voice: Optional[str] = "en-US-Chirp3-HD-Iapetus",
        customer_voice: Optional[str] = "en-US-Chirp3-HD-Zephyr",
        pause_between_utterances_ms: Optional[int] = 500,
        sample_rate_hertz: int = 32000,
    ) -> None:
        """
        Processes a single JSON transcript, generates stereo audio, and saves it.
        Args:
            transcript (dict): A dictionary containing the conversation transcript.
            audio_file_path (str): The path to save the generated audio file.
            language (str): The language code for speech synthesis.
            agent_voice (str): The voice model for the agent.
            customer_voice (str): The voice model for the customer.
            pause_between_utterances_ms (str): Milliseconds of silence between utterances.
            sample_rate_hertz (int): The sample rate in Hertz.
        """
        speech_client = speech.TextToSpeech(project_id=self.project_id)
        voice_overrides = {Roles.AGENT: agent_voice, Roles.CUSTOMER: customer_voice}
        if "entries" not in transcript or not isinstance(transcript["entries"], list):
            raise ValueError(
                "Transcript must be a dictionary containing an"
                "'entries' key with a list of utterances."
            )

        ##TO-DO: support different audio format selection
        audio_format = "wav"
        combined_audio = AudioSegment.empty()
        entries = transcript["entries"]
        for i, entry in enumerate(entries):
            text = entry.get("text")
            role_str = entry.get("role", "").upper()
            role_enum = Roles[role_str]

            voice = voice_overrides.get(role_enum) or VoiceRole[role_enum.name].value

            mono_audio_bytes = speech_client.synthesize(
                text=text,
                voice=voice,
                language_code=language,
                sample_rate_hertz=sample_rate_hertz,
            )

            mono_segment = AudioSegment.from_file(
                io.BytesIO(mono_audio_bytes), format=audio_format
            )
            silent_segment = AudioSegment.silent(
                duration=len(mono_segment), frame_rate=mono_segment.frame_rate
            )

            # Ensure both segments have the exact same length to prevent errors.
            min_len = min(len(mono_segment), len(silent_segment))
            mono_segment = mono_segment[:min_len]
            silent_segment = silent_segment[:min_len]

            # Define left and right channels based on the role.
            left_channel, right_channel = (
                (mono_segment, silent_segment)
                if role_enum == Roles.CUSTOMER
                else (silent_segment, mono_segment)
            )

            # Create the stereo audio segment.
            stereo_utterance = AudioSegment.from_mono_audiosegments(
                left_channel, right_channel
            )

            combined_audio += stereo_utterance

            if i < len(entries) - 1:
                combined_audio += AudioSegment.silent(
                    duration=pause_between_utterances_ms
                )

        if len(combined_audio) > 0:
            if audio_file_path.startswith("gs://"):
                # Create an in-memory binary stream to hold the audio data
                buffer = io.BytesIO()
                combined_audio.export(buffer, format=audio_format)
                buffer.seek(0)  # Rewind the buffer to the beginning before reading
                gcs_path = audio_file_path.replace("gs://", "")
                bucket_name, blob_name = gcs_path.split("/", 1)

                # Initialize the storage client and upload the in-memory file
                storage_client = storage.Gcs(
                    bucket_name=bucket_name, project_id=self.project_id
                )
                storage_client.upload_blob(
                    file_name=blob_name,
                    data=buffer.read(),
                    content_type=storage.ContentType.WAV,
                )
            else:
                # If it's not a GCS path, save it as a local file
                combined_audio.export(audio_file_path, format=audio_format)
            print(f"Audio successfully generated and saved to: {audio_file_path}")
        else:
            raise ValueError("Combined audio is empty.")

class RedactAudio:
    """
    A class to redact audio using STT and DLP
    """

    def __init__(
        self,
        project_id: str,
    ):
        """
        Initializes the GenerateAudio class.
        Args:
            project_id (str): The Google Cloud project ID.
        """

        raise NotImplementedError("RedactAudio is not yet implemented.")
# --- END OF ORIGINAL audio.py CODE ---

## 3. Using `GenerateAudio`

The `GenerateAudio` class is the core component for synthesizing speech into stereo audio. It expects a `project_id` during initialization.

### 3.1. Initializing `GenerateAudio`

Instantiate the class with your Google Cloud Project ID.

In [ ]:
YOUR_PROJECT_ID = "your-gcp-project-id" # <-- REPLACE WITH YOUR GOOGLE CLOUD PROJECT ID

audio_generator = GenerateAudio(project_id=YOUR_PROJECT_ID)
print(f"GenerateAudio instance created for project: {YOUR_PROJECT_ID}")

### 3.2. Generating a Single Audio File with `single()`

The `single()` method processes one transcript (a dictionary with an `entries` list) and generates a stereo audio file. The `role` in each entry (CUSTOMER or AGENT) determines which channel (left or right) the utterance is placed on.

**Transcript Format:**

The `transcript` argument should be a dictionary with an `entries` key, which is a list of utterances. Each utterance is a dictionary containing at least `"text"` and `"role"`.

```json
{
  "entries": [
    {"text": "Hello, how can I help you today?", "role": "AGENT"},
    {"text": "Yes, I have a question about my recent order.", "role": "CUSTOMER"}
  ]
}
```

**Parameters:**
*   `transcript` (dict): The conversation transcript.
*   `audio_file_path` (str): Local path or GCS path (e.g., `gs://your-bucket/file.wav`) to save the audio.
*   `language` (str, optional): Language code (default: `"en-US"`).
*   `agent_voice` (str, optional): Voice model for the agent (default: `"en-US-Chirp3-HD-Iapetus"`).
*   `customer_voice` (str, optional): Voice model for the customer (default: `"en-US-Chirp3-HD-Zephyr"`).
*   `pause_between_utterances_ms` (int, optional): Silence duration between utterances (default: `500` ms).
*   `sample_rate_hertz` (int, optional): Audio sample rate (default: `32000` Hz).

#### Example: Local Audio File Generation

In [ ]:
example_transcript_1 = {
    "entries": [
        {"text": "Hello, thank you for calling support. How may I help you?", "role": "AGENT"},
        {"text": "Hi. I'm having trouble logging into my account.", "role": "CUSTOMER"},
        {"text": "I see. Can you please confirm your username?", "role": "AGENT"},
        {"text": "It's J. Smith twenty twenty-three.", "role": "CUSTOMER"},
        {"text": "Thank you. Let me look that up for you.", "role": "AGENT"}
    ]
}

output_local_file = "conversation_stereo_1.wav"

try:
    audio_generator.single(
        transcript=example_transcript_1,
        audio_file_path=output_local_file,
        language="en-US",
        agent_voice="en-US-Wavenet-D", # Using a Wavenet voice for a more natural sound
        customer_voice="en-US-Wavenet-A",
        pause_between_utterances_ms=750 # Slightly longer pause
    )
    print(f"Generated local audio file: {output_local_file}")
    display(Audio(output_local_file, autoplay=False))
except Exception as e:
    print(f"Error generating audio: {e}")


#### Example: Google Cloud Storage (GCS) Output

To save to GCS, simply prefix your `audio_file_path` with `gs://` followed by your bucket name and blob path. Make sure the specified GCS bucket exists and your service account or user credentials have write permissions to it.

**Note:** Since we are using mock `storage.Gcs`, this will only print a message indicating the simulated upload. In a real scenario, the file would be uploaded to your specified GCS bucket.

In [ ]:
YOUR_GCS_BUCKET = "your-gcs-bucket-name" # <-- REPLACE WITH YOUR GCS BUCKET NAME

example_transcript_2 = {
    "entries": [
        {"text": "Good morning. This is the sales department.", "role": "AGENT"},
        {"text": "Hello. I'm interested in your new product line.", "role": "CUSTOMER"},
        {"text": "Excellent! What specifically caught your eye?", "role": "AGENT"}
    ]
}

output_gcs_path = f"gs://{YOUR_GCS_BUCKET}/sales_inquiry_stereo.wav"

try:
    audio_generator.single(
        transcript=example_transcript_2,
        audio_file_path=output_gcs_path,
        language="en-US"
    )
    print(f"Simulated GCS upload for: {output_gcs_path}")
except Exception as e:
    print(f"Error simulating GCS audio generation: {e}")

### 3.3. Bulk Audio Generation with `bulk()`

The `bulk()` method allows you to process a list of transcripts. It iterates through the list, calling `single()` for each transcript. This is useful for batch processing multiple conversations.

**Note:** For demonstration purposes, this example will overwrite the same output file repeatedly for simplicity. In a real application, you would typically use unique `audio_file_path` values for each transcript in the `bulk` call, e.g., by dynamically generating file names.

In [ ]:
bulk_transcripts = [
    {
        "entries": [
            {"text": "Hello again. Is there anything else I can assist you with?", "role": "AGENT"},
            {"text": "Yes, I need to update my shipping address.", "role": "CUSTOMER"}
        ]
    },
    {
        "entries": [
            {"text": "Welcome to our service. How are you today?", "role": "AGENT"},
            {"text": "I'm great, just calling to inquire about a promotion.", "role": "CUSTOMER"}
        ]
    }
]

# Note: In a real scenario, you'd generate unique paths, e.g., using a loop
# For this demo, we'll simulate processing by calling bulk on the same path multiple times
bulk_output_file_prefix = "bulk_conversation_output_"
output_files = []

print("Starting bulk audio generation...")
for i, transcript in enumerate(bulk_transcripts):
    current_output_file = f"{bulk_output_file_prefix}{i+1}.wav"
    try:
        audio_generator.single(transcript=transcript, audio_file_path=current_output_file)
        output_files.append(current_output_file)
        print(f"Generated {current_output_file}")
        display(Audio(current_output_file, autoplay=False))
    except Exception as e:
        print(f"Error generating audio for bulk transcript {i+1}: {e}")

print("\nBulk audio generation complete. The following files were created (or simulated for GCS):")
for f in output_files:
    print(f"- {f}")

## 4. `RedactAudio` (Not Implemented)

The `audio.py` module also defines a `RedactAudio` class, intended for redacting sensitive information from audio using Speech-to-Text (STT) and Data Loss Prevention (DLP).

As of the provided source code, this class is explicitly marked as `NotImplementedError` upon initialization, meaning its functionality is not yet available for use. This tutorial therefore focuses solely on the `GenerateAudio` capabilities.

In [ ]:
try:
    redaction_tool = RedactAudio(project_id=YOUR_PROJECT_ID)
except NotImplementedError as e:
    print(f"Caught expected error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## 5. Conclusion

This tutorial demonstrated how to use the `GenerateAudio` class from the `audio.py` module to synthesize stereo audio from conversational transcripts. You learned how to:

*   Set up your environment and mock internal dependencies.
*   Initialize the `GenerateAudio` class with your Google Cloud Project ID.
*   Use the `single()` method to generate stereo audio from a single transcript, saving it locally or to Google Cloud Storage.
*   Utilize the `bulk()` method for processing multiple transcripts.

This module provides a powerful way to create realistic conversational audio for various applications, such as training AI models, creating voice assistants, or generating accessible content. Remember to replace placeholder values like `YOUR_PROJECT_ID` and `YOUR_GCS_BUCKET` with your actual Google Cloud credentials and resources.